# Deep Dive: CUDA — Euclidean Norm (C vs CUDA Unified Memory variants)

**Group 3:** Matthew Fabregas, Kane Ibanez, Bowen Li, John Teoxon

**Specification (from our SIMD project):** compute the Euclidean norm of a double-precision vector `A` of length `n`

$$Z = \sqrt{\sum_{i=0}^{n-1} A_i^2}$$

**Data initialization:** `A[i] = sin(i * 0.001)` (double precision), **n = 2^24 = 16,777,216 elements** (128 MB).

**Variants Requirements :**
1.  C/C++ program version = answer key
2. CUDA program version using a grid-stride loop without prefetch and without mem advise = we used unified memory
3. CUDA program version using a grid-stride loop with prefetching but without page creation and without mem advise
4. CUDA program version using a grid-stride loop with prefetch, with page creation but without mem advise
5. CUDA program version using a grid-stride loop with prefetch, with page creation and with mem advise

Each Kernal executed 30 times for block sizes:256, 512 and 1024. This is to "jump start" process to compare "cold start" from "hot start", then we compare it to answer key

For the grid size, we set up the main runs to only launch as many blocks as the GPU can handle at once (number of SMs × resident blocks per SM). Because of this, each thread actually loops through a bunch of elements using the grid-stride loop. We also made a separate cell to compare this against the "maximum blocks" approach (n / blockSize), which is basically just one thread per element.

**NOTE:** Use T4 GPU to run this or we won't have enough process power

In [ ]:
!nvidia-smi
!nvcc --version
!rm -f results.csv

Mon Nov  9 13:48:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Version 1 — C program (baseline and sanity-check answer key)

For this one, we basically used the same algorithm as norm_c from our SIMD project. It just runs a sequential loop that adds A[i] * A[i] and then takes the sqrt. We initialized the vector with A[i] = sin(i * 0.001). We made sure to do one untimed warm-up call first, and then we ran 30 timed calls (we only timed the actual function call itself using clock_gettime). The final Z result from this is what we use as the answer key to check if all our CUDA versions are correct.

In [ ]:
%%writefile euclid_c.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define RUNS 30

double norm_c(const double* A, long long n) {
    double sum = 0.0;
    for (long long i = 0; i < n; i++)
        sum += A[i] * A[i];
    return sqrt(sum);
}

static double now_ms(void) {
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1000.0 + t.tv_nsec / 1e6;
}

int main(int argc, char** argv) {
    int p = (argc > 1) ? atoi(argv[1]) : 24;
    long long n = 1LL << p;
    double* A = (double*)malloc(n * sizeof(double));
    for (long long i = 0; i < n; i++)
        A[i] = sin(i * 0.001);

    double z = norm_c(A, n); // warm-up (untimed)
    double total = 0, mn = 1e300, mx = 0, first = 0;
    for (int r = 0; r < RUNS; r++) {
        double t1 = now_ms();
        z = norm_c(A, n);
        double ms = now_ms() - t1;
        if (r == 0) first = ms;
        total += ms;
        if (ms < mn) mn = ms;
        if (ms > mx) mx = ms;
    }
    double avg = total / RUNS;

    printf("VERSION 1: C program (baseline / answer key)\n");
    printf("n = 2^%d = %lld elements, A[i] = sin(i*0.001)\n", p, n);
    printf("Z = %.10f\n", z);
    printf("Execution time over %d runs: avg = %.4f ms | min = %.4f ms | max = %.4f ms\n", RUNS, avg, mn, mx);

    FILE* f = fopen("results.csv", "a");
    if (f) {
        fprintf(f, "C,-,-,-,%d,%.6f,%.6f,%.6f,%.6f,%.12f,%.12f,PASS\n", p, first, avg, mn, mx, z, z);
        fclose(f);
    }
    free(A);
    return 0;
}

Writing euclid_c.c


In [ ]:
!gcc -O2 -o euclid_c euclid_c.c -lm
!./euclid_c 24

VERSION 1: C program (baseline / answer key)
n = 2^24 = 16777216 elements, A[i] = sin(i*0.001)
Z = 2896.2749920235
Execution time over 30 runs: avg = 31.2619 ms | min = 24.6707 ms | max = 46.3645 ms


## Shared CUDA code (kernel and helpers used for all 4 CUDA versions)

We made all four CUDA versions run the exact same kernel, so the only thing that actually changes between them is how we handle the Unified Memory.

Kernel (norm_kernel) = We went with a reduction for this, since we need to add every element into a single sum.

**Setup:**

- Grid-stride loop: Each thread basically squares and adds its assigned elements (index, index + stride, index + 2*stride, etc.) into a private register we called local. The stride here is just the total number of threads in the grid.

- Shared memory: Each block takes all those local values from its threads and adds them together using a tree reduction (making sure to use __syncthreads() between steps). The shared array gets one double per thread (blockSize * 8 bytes, which we pass in when we launch the kernel).

- atomicAdd: Finally, we just have thread 0 of each block add the block's total directly into the global sum. If we didn't use an atomic here, the blocks would just overwrite each other. Additionally, doing it this way means we only need one atomic per block instead of one for every single element.

After the kernel finishes, the host just takes the sqrt(sum).

**Grid size (gridMode):**

- gridMode 1 (main runs): We set it up so numBlocks = number of SMs × resident blocks per SM (we grabbed this using cudaOccupancyMaxActiveBlocksPerMultiprocessor). Basically, this is the exact number of blocks the GPU can handle at the same time. Since this makes our grid way smaller than n, every single thread actually loops through about n / (numBlocks × blockSize) elements. Because we're running this on a T4 GPU (which has 40 SMs and maxes out at 1024 threads per SM), we end up with 160, 80, or 40 blocks depending on if our block size is 256, 512, or 1024. That comes out to around 410 elements processed per thread.

- gridMode 0 (comparison only): We did numBlocks = (n + blockSize - 1) / blockSize, which gives us the absolute maximum number of blocks so it's just one element per thread. When we launch it like this, the stride is ≥ n, meaning the grid-stride loop only runs once for each thread. To put it in perspective, at a block size of 256, this launches a massive 65,536 blocks.

Correctness check: For the reference Z, we just used the exact same C function from Version 1. Since the GPU adds everything up in a slightly different order, the last few digits can sometimes be off. Because of that, we mark a result as CORRECT as long as the relative difference from the C version is at most 1e-9 (SIMD project tolerance).

Timing: For every single one of the 30 runs, we made sure to reset sum using cudaMemset first. After that, we strictly just timed the kernel itself using CUDA events. We decided to report run 1 separately since that first run always takes a hit from any initial page migration.

CUDA 12 vs 13: We noticed CUDA 13 actually changed cudaMemPrefetchAsync and cudaMemAdvise so they require a cudaMemLocation now. To handle this, we just wrote some small wrappers below that automatically pick the right call depending on whichever toolkit version Colab happens to be running.

Program arguments: We run it with ./program blockSize [gridMode] [power] (by default, we set it to 256, 1, and 24).

In [ ]:
%%writefile common.cuh
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

#define RUNS 30

#define CUDA_CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
    printf("CUDA error %s at line %d: %s\n", #x, __LINE__, cudaGetErrorString(e)); exit(1); } } while (0)

// ---------------- kernels ----------------

// Sum of squares with a grid-stride loop, shared-memory block reduction, one atomicAdd per block.
// Shared memory size (blockDim.x doubles) is given as the 3rd launch parameter.
__global__ void norm_kernel(const double* A, long long n, double* sum) {
    extern __shared__ double s[];
    long long index  = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;

    double local = 0.0;
    for (long long i = index; i < n; i += stride)
        local += A[i] * A[i];

    s[threadIdx.x] = local;
    __syncthreads();
    for (int k = blockDim.x / 2; k > 0; k >>= 1) {   // blockDim is 256/512/1024 (power of 2)
        if (threadIdx.x < k)
            s[threadIdx.x] += s[threadIdx.x + k];
        __syncthreads();
    }
    if (threadIdx.x == 0)
        atomicAdd(sum, s[0]);
}

// Initializes A on the GPU (used by the "page creation" versions), grid-stride loop.
__global__ void init_kernel(double* A, long long n) {
    long long index  = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    long long stride = (long long)blockDim.x * gridDim.x;
    for (long long i = index; i < n; i += stride)
        A[i] = sin(i * 0.001);
}

// ---------------- host helpers ----------------

// Same initialization as Version 1 (CPU)
inline void init_host(double* A, long long n) {
    for (long long i = 0; i < n; i++)
        A[i] = sin(i * 0.001);
}

// C answer key: same values and same order as norm_c in Version 1 (not timed)
inline double reference_norm(long long n) {
    double sum = 0.0;
    for (long long i = 0; i < n; i++) {
        double a = sin(i * 0.001);
        sum += a * a;
    }
    return sqrt(sum);
}

// gridMode 0: one element per thread (maximum number of blocks, the loop runs once)
// gridMode 1: blocks that fit on the GPU at once (SMs x resident blocks per SM), every thread loops
inline int pick_grid(long long n, int blockSize, int gridMode) {
    long long oneEach = (n + blockSize - 1) / blockSize;
    if (gridMode == 0) return (int)oneEach;
    int dev = 0, sms = 0, perSM = 0;
    CUDA_CHECK(cudaGetDevice(&dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&sms, cudaDevAttrMultiProcessorCount, dev));
    CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&perSM, norm_kernel, blockSize,
                                                             blockSize * sizeof(double)));
    long long g = (long long)sms * perSM;
    return (int)(g < oneEach ? g : oneEach);
}

inline void prefetch_to_gpu(const void* p, size_t bytes, int dev) {
#if CUDART_VERSION >= 13000
    cudaMemLocation loc = {};
    loc.type = cudaMemLocationTypeDevice;
    loc.id = dev;
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, loc, 0, 0));
#else
    CUDA_CHECK(cudaMemPrefetchAsync(p, bytes, dev, 0));
#endif
}

inline void advise_gpu(const void* p, size_t bytes, cudaMemoryAdvise advice, int dev) {
#if CUDART_VERSION >= 13000
    cudaMemLocation loc = {};
    loc.type = cudaMemLocationTypeDevice;
    loc.id = dev;
    CUDA_CHECK(cudaMemAdvise(p, bytes, advice, loc));
#else
    CUDA_CHECK(cudaMemAdvise(p, bytes, advice, dev));
#endif
}

// Runs the kernel RUNS times, times each run, checks correctness, prints and appends to results.csv
inline void run_and_report(const char* name, const char* label, const double* A, long long n,
                           int p, int blockSize, int gridMode, double* sum) {
    int numBlocks = pick_grid(n, blockSize, gridMode);
    size_t shmem = blockSize * sizeof(double);
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    double times[RUNS];
    for (int r = 0; r < RUNS; r++) {
        CUDA_CHECK(cudaMemset(sum, 0, sizeof(double)));   // reset sum (not timed)
        CUDA_CHECK(cudaEventRecord(start));
        norm_kernel<<<numBlocks, blockSize, shmem>>>(A, n, sum);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        CUDA_CHECK(cudaGetLastError());
        float ms = 0;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        times[r] = ms;
    }
    CUDA_CHECK(cudaDeviceSynchronize());
    double z = sqrt(*sum);                                   // result of the last run

    double total = 0, mn = 1e300, mx = 0, warm = 0;
    for (int r = 0; r < RUNS; r++) {
        total += times[r];
        if (times[r] < mn) mn = times[r];
        if (times[r] > mx) mx = times[r];
        if (r > 0) warm += times[r];
    }
    double avg = total / RUNS;
    warm /= (RUNS - 1);

    double ref = reference_norm(n);
    double rel = fabs(z - ref) / fabs(ref);
    bool ok = rel <= 1e-9;
    double perThread = (double)n / ((double)numBlocks * blockSize);

    printf("%s\n", name);
    printf("n = 2^%d = %lld | block size = %d | blocks = %d (gridMode %d, %s) | elements per thread = %.1f\n",
           p, n, blockSize, numBlocks, gridMode, gridMode == 0 ? "one element per thread" : "blocks that fit on the GPU",
           perThread);
    printf("Kernel time over %d runs: avg = %.4f ms | min = %.4f ms | max = %.4f ms\n", RUNS, avg, mn, mx);
    printf("Run 1 = %.4f ms | avg of runs 2-%d = %.4f ms | bandwidth (avg 2-%d) = %.1f GB/s\n",
           times[0], RUNS, warm, RUNS, (double)n * 8.0 / (warm * 1e6));
    printf("Z (CUDA) = %.10f | Z (C answer key) = %.10f | rel diff = %.2e -> %s\n\n",
           z, ref, rel, ok ? "CORRECT" : "WRONG");

    FILE* f = fopen("results.csv", "a");
    if (f) {
        fprintf(f, "%s,%d,%d,%d,%d,%.6f,%.6f,%.6f,%.6f,%.12f,%.12f,%s\n",
                label, blockSize, gridMode, numBlocks, p, times[0], avg, mn, mx, z, ref, ok ? "PASS" : "FAIL");
        fclose(f);
    }
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
}

// args: blockSize [gridMode, default 1] [power p, default 24]
inline void parse_args(int argc, char** argv, int* blockSize, int* gridMode, int* p) {
    *blockSize = (argc > 1) ? atoi(argv[1]) : 256;
    *gridMode  = (argc > 2) ? atoi(argv[2]) : 1;
    *p         = (argc > 3) ? atoi(argv[3]) : 24;
    if (*blockSize < 32 || *blockSize > 1024 || (*blockSize & (*blockSize - 1)) != 0) {
        printf("blockSize must be a power of two between 32 and 1024\n");
        exit(1);
    }
}

Writing common.cuh


## Version 2 — CUDA Unified Memory (grid-stride loop, no prefetch, no page creation, no mem advise)

For this version, we allocated A using cudaMallocManaged and initialized it over on the CPU, which means the pages are originally created in the CPU's memory. We didn't do anything extra here—no prefetching or anything like that. So what happens is, the second the kernel actually touches A for the first time, the GPU hits a page-fault and has to migrate those pages over to the GPU on demand. Because of this, all that initial migration overhead gets dumped straight into run 1 of the kernel.

In [ ]:
%%writefile cuda_unified.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    init_host(A, n);   // pages created on the CPU; no prefetch, no advise

    run_and_report("VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)",
                   "CUDA Unified", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_unified.cu


In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_unified cuda_unified.cu
!for b in 256 512 1024; do ./cuda_unified $b 1 24; done

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 1.5049 ms | min = 0.5243 ms | max = 29.8330 ms
Run 1 = 29.8330 ms | avg of runs 2-30 = 0.5281 ms | bandwidth (avg 2-30) = 254.2 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.97e-14 -> CORRECT

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 512 | blocks = 80 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 1.4769 ms | min = 0.5223 ms | max = 29.0670 ms
Run 1 = 29.0670 ms | avg of runs 2-30 = 0.5255 ms | bandwidth (avg 2-30) = 255.4 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.96e-14 -> CORRECT

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 1

## Version 3 — CUDA Unified Memory + prefetch (no page creation, no mem advise)

This one is the same as Version 2 (A initialized on the CPU), but this time we made sure to call cudaMemPrefetchAsync(A, bytes, device) right before we launch the kernel. By doing this, the entire vector gets copied over to the GPU in massive chunks before the kernel even starts running. Because it's already there, the kernel itself doesn't have to deal with any page-faults at all.

In [ ]:
%%writefile cuda_prefetch.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    init_host(A, n);                        // pages created on the CPU
    prefetch_to_gpu(A, bytes, device);      // migrate A to the GPU before the kernel
    prefetch_to_gpu(sum, sizeof(double), device);
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 3: CUDA Prefetch (no page creation, no mem advise)",
                   "CUDA Prefetch", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_prefetch.cu


In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_prefetch cuda_prefetch.cu
!for b in 256 512 1024; do ./cuda_prefetch $b 1 24; done

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5310 ms | min = 0.5278 ms | max = 0.5479 ms
Run 1 = 0.5374 ms | avg of runs 2-30 = 0.5308 ms | bandwidth (avg 2-30) = 252.9 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.97e-14 -> CORRECT

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 512 | blocks = 80 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5266 ms | min = 0.5222 ms | max = 0.5714 ms
Run 1 = 0.5305 ms | avg of runs 2-30 = 0.5265 ms | bandwidth (avg 2-30) = 254.9 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.96e-14 -> CORRECT

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 1024 | blocks = 4

## Version 4 — CUDA Unified Memory + prefetch + page creation (no mem advise)

How we defined page creation: When you use cudaMallocManaged, it really just reserves the addresses at first, the actual physical pages aren't created until the memory is touched. For this run, we made sure to create the pages directly in the GPU memory. To pull this off, we take the still-empty A array and prefetch it straight to the GPU, and then we initialize it using a GPU kernel (we named it init_kernel, which uses the exact same sin(i * 0.001) formula and another grid-stride loop). Since the CPU never actually writes any of the data, nothing has to be migrated from the CPU over to the GPU.

In [ ]:
%%writefile cuda_pages.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    prefetch_to_gpu(A, bytes, device);      // prefetch the empty buffer to the GPU
    prefetch_to_gpu(sum, sizeof(double), device);
    init_kernel<<<pick_grid(n, blockSize, 1), blockSize>>>(A, n);   // page creation on the GPU
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 4: CUDA Prefetch + page creation (no mem advise)",
                   "CUDA Prefetch+Page creation", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_pages.cu


In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_pages cuda_pages.cu
!for b in 256 512 1024; do ./cuda_pages $b 1 24; done

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5293 ms | min = 0.5246 ms | max = 0.5456 ms
Run 1 = 0.5407 ms | avg of runs 2-30 = 0.5289 ms | bandwidth (avg 2-30) = 253.8 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.97e-14 -> CORRECT

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 512 | blocks = 80 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5248 ms | min = 0.5222 ms | max = 0.5366 ms
Run 1 = 0.5366 ms | avg of runs 2-30 = 0.5244 ms | bandwidth (avg 2-30) = 255.9 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.96e-14 -> CORRECT

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 1024 | blocks = 40 (gri

## Version 5 — CUDA Unified Memory + prefetch + page creation + mem advise

same as Version 4, but we used cudaMemAdvise hints:

We set cudaMemAdviseSetPreferredLocation to the GPU for both A and sum. We made sure to do this before the prefetch step so the driver knows to keep those pages locked in the GPU memory.

We also used cudaMemAdviseSetReadMostly for A, but we set this one after it was already initialized. We did this because our norm kernel only ever needs to read A anyway (it is just a source operand).

In [ ]:
%%writefile cuda_advise.cu
#include "common.cuh"

int main(int argc, char** argv) {
    int blockSize, gridMode, p;
    parse_args(argc, argv, &blockSize, &gridMode, &p);
    long long n = 1LL << p;
    size_t bytes = n * sizeof(double);
    int device = -1;
    CUDA_CHECK(cudaGetDevice(&device));

    double *A, *sum;
    CUDA_CHECK(cudaMallocManaged(&A, bytes));
    CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));

    advise_gpu(A, bytes, cudaMemAdviseSetPreferredLocation, device);           // mem advise
    advise_gpu(sum, sizeof(double), cudaMemAdviseSetPreferredLocation, device);
    prefetch_to_gpu(A, bytes, device);                                         // prefetch
    prefetch_to_gpu(sum, sizeof(double), device);
    init_kernel<<<pick_grid(n, blockSize, 1), blockSize>>>(A, n);              // page creation on the GPU
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    advise_gpu(A, bytes, cudaMemAdviseSetReadMostly, device);                  // A is read-only from now on
    CUDA_CHECK(cudaDeviceSynchronize());

    run_and_report("VERSION 5: CUDA Prefetch + page creation + mem advise",
                   "CUDA Prefetch+Page creation+memadvise", A, n, p, blockSize, gridMode, sum);

    CUDA_CHECK(cudaFree(A));
    CUDA_CHECK(cudaFree(sum));
    return 0;
}

Writing cuda_advise.cu


In [ ]:
!nvcc -O2 -arch=sm_75 -o cuda_advise cuda_advise.cu
!for b in 256 512 1024; do ./cuda_advise $b 1 24; done

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5318 ms | min = 0.5260 ms | max = 0.5509 ms
Run 1 = 0.5509 ms | avg of runs 2-30 = 0.5312 ms | bandwidth (avg 2-30) = 252.7 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.99e-14 -> CORRECT

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 512 | blocks = 80 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5316 ms | min = 0.5253 ms | max = 0.5673 ms
Run 1 = 0.5427 ms | avg of runs 2-30 = 0.5312 ms | bandwidth (avg 2-30) = 252.7 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.97e-14 -> CORRECT

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 1024 | blocks = 40 (gridMode 1, 

## Block size and number of blocks: "blocks that fit on the GPU" vs "maximum blocks"

For guide question "(a)How does block size affect execution time (observing various elements and using max blocks)?  Which block size will you recommend?".

We needed to figure out how block size affects execution time when we use the maximum number of blocks. To test this, we set up this cell to run all four versions again, but this time using gridMode 0: numBlocks = (n + blockSize - 1) / blockSize. This assigns exactly one element per thread (which is around 65,536, 32,768, or 16,384 blocks depending on the block size). We can compare these times directly with our main runs from earlier where we used gridMode 1 (it only launched 160, 80, or 40 blocks on our T4).

Because we're using gridMode 0 here, the grid-stride loop really only runs once per thread. And since we're launching a massive amount of blocks, every single one of them has to do its own shared-memory reduction and atomicAdd.

In [ ]:
%%bash
for v in cuda_unified cuda_prefetch cuda_pages cuda_advise; do
  for b in 256 512 1024; do ./$v $b 0 24; done
done

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 65536 (gridMode 0, one element per thread) | elements per thread = 1.0
Kernel time over 30 runs: avg = 4.2808 ms | min = 1.8536 ms | max = 74.6038 ms
Run 1 = 74.6038 ms | avg of runs 2-30 = 1.8559 ms | bandwidth (avg 2-30) = 72.3 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.22e-14 -> CORRECT

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 512 | blocks = 32768 (gridMode 0, one element per thread) | elements per thread = 1.0
Kernel time over 30 runs: avg = 3.0069 ms | min = 1.8719 ms | max = 35.4271 ms
Run 1 = 35.4271 ms | avg of runs 2-30 = 1.8889 ms | bandwidth (avg 2-30) = 71.1 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.93e-14 -> CORRECT

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 |

## Other vector sizes: 2^20 and 2^26 (same sizes as our SIMD project)

We also needed to see how the block size affects the execution time when we change up the number of elements. Because of that, this cell basically repeats everything we just did, but for n = 2^20 (1,048,576 elements, 8 MB) and n = 2^26 (67,108,864 elements, 512 MB).

It runs the C baseline again, and then goes through all four CUDA versions using block sizes of 256, 512, and 1024. We made sure to test this with both launch setups too (gridMode 1 for just the blocks that actually fit on the GPU, and gridMode 0 for the maximum blocks where it's one element per thread). We kept the exact same data initialization, the 30 runs, and the correctness checks from our main runs. You can find all the summary tables printed out down in the Results summary section.

**Note**: T4 GPU has tendency to stop midway because of too much processing for some reason

In [ ]:
%%bash
for p in 20 26; do
  echo "################ n = 2^$p ################"
  ./euclid_c $p
  for v in cuda_unified cuda_prefetch cuda_pages cuda_advise; do
    for g in 1 0; do
      for b in 256 512 1024; do ./$v $b $g $p; done
    done
  done
done

################ n = 2^20 ################
VERSION 1: C program (baseline / answer key)
n = 2^20 = 1048576 elements, A[i] = sin(i*0.001)
Z = 724.2481457528
Execution time over 30 runs: avg = 1.5324 ms | min = 1.3297 ms | max = 2.9236 ms
VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^20 = 1048576 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 25.6
Kernel time over 30 runs: avg = 0.1630 ms | min = 0.0410 ms | max = 3.6500 ms
Run 1 = 3.6500 ms | avg of runs 2-30 = 0.0428 ms | bandwidth (avg 2-30) = 196.1 GB/s
Z (CUDA) = 724.2481457528 | Z (C answer key) = 724.2481457528 | rel diff = 1.33e-14 -> CORRECT

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^20 = 1048576 | block size = 512 | blocks = 80 (gridMode 1, blocks that fit on the GPU) | elements per thread = 25.6
Kernel time over 30 runs: avg = 0.1649 ms | min = 0.0428 ms | max = 3.6618 ms
Run 1 = 3.6618 ms | avg of runs 2-30 = 

## Boundary test (same sizes as our SIMD project test)

This part just checks how the CUDA kernel handles really small and odd sizes (like n = 1, 3, 1000, 1001, 1002, and 1003) where the number of elements doesn't divide perfectly by the block size or the grid. Because of how it's set up, the grid-stride loop's i < n check actually handles all those leftover elements automatically (our old SIMD versions needed its own tail loop just to deal with that). We just ran this using our main launch setup (gridMode 1 with a block size of 256).

In [ ]:
%%writefile boundary.cu
#include "common.cuh"

int main() {
    long long sizes[6] = {1, 3, 1000, 1001, 1002, 1003};
    int blockSize = 256, fails = 0;
    printf("boundary test (block size %d, gridMode 1)\n", blockSize);
    for (int k = 0; k < 6; k++) {
        long long n = sizes[k];
        double *A, *sum;
        CUDA_CHECK(cudaMallocManaged(&A, n * sizeof(double)));
        CUDA_CHECK(cudaMallocManaged(&sum, sizeof(double)));
        init_host(A, n);
        int grid = pick_grid(n, blockSize, 1);
        CUDA_CHECK(cudaMemset(sum, 0, sizeof(double)));
        norm_kernel<<<grid, blockSize, blockSize * sizeof(double)>>>(A, n, sum);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());
        double z = sqrt(*sum), ref = reference_norm(n);
        bool ok = fabs(z - ref) <= 1e-9 * fabs(ref);   // also works when ref = 0 (n = 1)
        if (!ok) fails++;
        printf("n = %4lld  blocks = %d  C = %.10f  CUDA = %.10f  %s\n", n, grid, ref, z, ok ? "correct" : "WRONG");
        CUDA_CHECK(cudaFree(A));
        CUDA_CHECK(cudaFree(sum));
    }
    printf(fails ? "%d FAILED\n" : "all correct\n", fails);
    return fails;
}

Writing boundary.cu


In [ ]:
!nvcc -O2 -arch=sm_75 -o boundary boundary.cu
!./boundary

boundary test (block size 256, gridMode 1)
n =    1  blocks = 1  C = 0.0000000000  CUDA = 0.0000000000  correct
n =    3  blocks = 1  C = 0.0022360667  CUDA = 0.0022360667  correct
n = 1000  blocks = 4  C = 16.5021720497  CUDA = 16.5021720497  correct
n = 1001  blocks = 4  C = 16.5236120681  CUDA = 16.5236120681  correct
n = 1002  blocks = 4  C = 16.5450517701  CUDA = 16.5450517701  correct
n = 1003  blocks = 4  C = 16.5664911318  CUDA = 16.5664911318  correct
all correct


## Results summary (2^24 elements)

We collected all these numbers from the runs we did above. To calculate the speedup, we just took the C average time and divided it by the version average time (making sure to use the exact same Colab machine and vector size so it's a fair comparison).

The very first table shows our main runs at 2^24 (using gridMode 1). After that, the second table compares both of our launch setups at 2^24 side-by-side. Finally, the last few tables break down all the results for 2^20, 2^24, and 2^26.

**NOTE:** the data for x86-64, XMM, and YMM actually came from our old SIMD project (ran in Visual Studio on Windows), so we just added those straight into the README separately.

In [ ]:
import pandas as pd
cols = ['version','block','gridMode','blocks','power','run1_ms','avg_ms','min_ms','max_ms','Z','Z_ref','correct']
df = pd.read_csv('results.csv', names=cols, dtype={'block': str, 'gridMode': str, 'blocks': str})
df = df[df['power'] == 24]
c_avg = df[df['version'] == 'C']['avg_ms'].iloc[-1]
df['speedup_vs_C'] = (c_avg / df['avg_ms']).round(2)
pd.set_option('display.width', 200)
show = ['version','block','blocks','run1_ms','avg_ms','min_ms','max_ms','speedup_vs_C','correct']

print('Main runs (gridMode 1: blocks that fit on the GPU, every thread loops)')
main = df[(df['version'] == 'C') | (df['gridMode'] == '1')]
print(main[show].round(4).to_string(index=False))

print('\nAverage kernel time (ms): gridMode 0 (n / blockSize blocks) vs gridMode 1 (blocks that fit on the GPU)')
cuda = df[df['version'] != 'C'].copy()
cuda['block'] = cuda['block'].astype(int)
pivot = cuda.pivot_table(index='version', columns=['gridMode', 'block'], values='avg_ms', aggfunc='last')
order = ['CUDA Unified', 'CUDA Prefetch', 'CUDA Prefetch+Page creation', 'CUDA Prefetch+Page creation+memadvise']
print(pivot.reindex(order).round(4).to_string())

# ---- all sizes (2^20, 2^24, 2^26): average kernel time and speedup vs C on the same size
print('\nAll sizes: average kernel time in ms (speedup vs C of the same size in brackets)')
allp = pd.read_csv('results.csv', names=cols, dtype={'block': str, 'gridMode': str, 'blocks': str})
for p in sorted(allp['power'].unique()):
    d = allp[allp['power'] == p]
    c = d[d['version'] == 'C']['avg_ms']
    if c.empty:
        continue
    c = c.iloc[-1]
    cu = d[d['version'] != 'C'].copy()
    cu['block'] = cu['block'].astype(int)
    cu['cell'] = cu['avg_ms'].map('{:.4f}'.format) + ' (' + (c / cu['avg_ms']).map('{:.1f}x'.format) + ')'
    piv = cu.pivot_table(index='version', columns=['gridMode', 'block'], values='cell', aggfunc='last')
    print(f'\nn = 2^{p} | C avg = {c:.4f} ms | all correct: {(d["correct"] == "PASS").all()}')
    print(piv.reindex(order).to_string())

Main runs (gridMode 1: blocks that fit on the GPU, every thread loops)
                              version block blocks  run1_ms  avg_ms  min_ms  max_ms  speedup_vs_C correct
                                    C     -      -  24.6707 31.2619 24.6707 46.3645          1.00    PASS
                         CUDA Unified   256    160  29.8330  1.5049  0.5243 29.8330         20.77    PASS
                         CUDA Unified   512     80  29.0670  1.4769  0.5223 29.0670         21.17    PASS
                         CUDA Unified  1024     40  27.7745  1.4333  0.5223 27.7745         21.81    PASS
                        CUDA Prefetch   256    160   0.5374  0.5310  0.5278  0.5479         58.87    PASS
                        CUDA Prefetch   512     80   0.5305  0.5266  0.5222  0.5714         59.36    PASS
                        CUDA Prefetch  1024     40   0.5386  0.5290  0.5254  0.5386         59.10    PASS
          CUDA Prefetch+Page creation   256    160   0.5407  0.5293  0.5246  0.54

## Profiling — Nsight Systems (nsys), all CUDA versions

We stick with a block size of 256 and our main launch setup (gridMode 1). We used the --stats=true flag so it prints out all the essential information such as the kernel summary, the Unified Memory transfer summaries (cuda_gpu_mem_time_sum and cuda_gpu_mem_size_sum), the Unified Memory totals (which includes the um_total_sum and page-fault counts), and the CUDA API summary.

We expect to see a bunch of page faults and small transfers for Version 2, exactly 64 huge transfers for Version 3, and absolutely no Host-to-Device transfers for Versions 4 and 5.

**NOTE:** running nsys slows the program down a bit for some reason, so the times in the table may be inaccurate for benchmarking.

In [ ]:
!nsys profile -o nsys_cuda_unified --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_unified 256 1 24

VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 2.3822 ms | min = 0.5367 ms | max = 55.6944 ms
Run 1 = 55.6944 ms | avg of runs 2-30 = 0.5439 ms | bandwidth (avg 2-30) = 246.8 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.99e-14 -> CORRECT

Generating '/tmp/nsys-report-5a76.qdstrm'
[ 1/11] [========================100%] nsys_cuda_unified.nsys-rep
[ 2/11] [========================100%] nsys_cuda_unified.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_unified.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)        Name     
 --------  ---------------  ---------  ------------  ------------  ------

In [ ]:
!nsys profile -o nsys_cuda_prefetch --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_prefetch 256 1 24

VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5414 ms | min = 0.5351 ms | max = 0.5543 ms
Run 1 = 0.5543 ms | avg of runs 2-30 = 0.5409 ms | bandwidth (avg 2-30) = 248.1 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.99e-14 -> CORRECT

Generating '/tmp/nsys-report-e92e.qdstrm'
[ 1/11] [========================100%] nsys_cuda_prefetch.nsys-rep
[ 2/11] [========================100%] nsys_cuda_prefetch.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_prefetch.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  -------- 

In [ ]:
!nsys profile -o nsys_cuda_pages --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_pages 256 1 24

VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5439 ms | min = 0.5365 ms | max = 0.5656 ms
Run 1 = 0.5595 ms | avg of runs 2-30 = 0.5434 ms | bandwidth (avg 2-30) = 247.0 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.99e-14 -> CORRECT

Generating '/tmp/nsys-report-c049.qdstrm'
[ 1/11] [========================100%] nsys_cuda_pages.nsys-rep
[ 2/11] [========================100%] nsys_cuda_pages.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_pages.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  ----------

In [ ]:
!nsys profile -o nsys_cuda_advise --force-overwrite true --stats=true --cuda-um-gpu-page-faults=true --cuda-um-cpu-page-faults=true ./cuda_advise 256 1 24

VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 0.5358 ms | min = 0.5307 ms | max = 0.5620 ms
Run 1 = 0.5620 ms | avg of runs 2-30 = 0.5349 ms | bandwidth (avg 2-30) = 250.9 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.99e-14 -> CORRECT

Generating '/tmp/nsys-report-4f0b.qdstrm'
[ 1/11] [========================100%] nsys_cuda_advise.nsys-rep
[ 2/11] [========================100%] nsys_cuda_advise.sqlite
[ 3/11] Executing 'nvtx_sum' stats report
SKIPPED: /content/nsys_cuda_advise.sqlite does not contain NV Tools Extension (NVTX) data.
[ 4/11] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)      Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ------------  --------  ----------

## Profiling — Nsight Compute (ncu), all CUDA versions

For the ncu profiling on all the CUDA versions, we just profiled a single norm_kernel launch for each version using -k norm_kernel --launch-count 1. We stuck with the 256 block size and gridMode 1 for this.  

**NOTE:** The kernel times printed on the ff cells are gonna look way slower than normal. This is because the profiler replays the kernel multiple times and locks the GPU clock speed, so we got all our actual benchmark numbers from the cells before/above instead of using this profiling step.

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_unified 256 1 24

==PROF== Connected to process 1582 (/content/cuda_unified)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 2: CUDA Unified (no prefetch, no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 86.7540 ms | min = 0.6516 ms | max = 2579.2446 ms
Run 1 = 2579.2446 ms | avg of runs 2-30 = 0.8060 ms | bandwidth (avg 2-30) = 166.5 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.96e-14 -> CORRECT

==PROF== Disconnected from process 1582
[1582] cuda_unified@127.0.0.1
  norm_kernel(const double *, long long, double *) (160, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency         

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_prefetch 256 1 24

==PROF== Connected to process 1646 (/content/cuda_prefetch)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 3: CUDA Prefetch (no page creation, no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 75.7681 ms | min = 0.6533 ms | max = 2252.2439 ms
Run 1 = 2252.2439 ms | avg of runs 2-30 = 0.7172 ms | bandwidth (avg 2-30) = 187.1 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.97e-14 -> CORRECT

==PROF== Disconnected from process 1646
[1646] cuda_prefetch@127.0.0.1
  norm_kernel(const double *, long long, double *) (160, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  G

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_pages 256 1 24

==PROF== Connected to process 1709 (/content/cuda_pages)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 4: CUDA Prefetch + page creation (no mem advise)
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 73.7222 ms | min = 0.6475 ms | max = 2190.5645 ms
Run 1 = 2190.5645 ms | avg of runs 2-30 = 0.7276 ms | bandwidth (avg 2-30) = 184.5 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 4.00e-14 -> CORRECT

==PROF== Disconnected from process 1709
[1709] cuda_pages@127.0.0.1
  norm_kernel(const double *, long long, double *) (160, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz      

In [ ]:
!ncu -k norm_kernel --launch-count 1 --set basic ./cuda_advise 256 1 24

==PROF== Connected to process 1765 (/content/cuda_advise)
==PROF== Profiling "norm_kernel": 0%....50%....100% - 10 passes
VERSION 5: CUDA Prefetch + page creation + mem advise
n = 2^24 = 16777216 | block size = 256 | blocks = 160 (gridMode 1, blocks that fit on the GPU) | elements per thread = 409.6
Kernel time over 30 runs: avg = 74.7265 ms | min = 0.6589 ms | max = 2221.6367 ms
Run 1 = 2221.6367 ms | avg of runs 2-30 = 0.6951 ms | bandwidth (avg 2-30) = 193.1 GB/s
Z (CUDA) = 2896.2749920236 | Z (C answer key) = 2896.2749920235 | rel diff = 3.99e-14 -> CORRECT

==PROF== Disconnected from process 1765
[1765] cuda_advise@127.0.0.1
  norm_kernel(const double *, long long, double *) (160, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz       